# 模块 3 节点壳：Navigator ROS 适配示例

对应 `navigator_node.py`，仅演示 `TrackArray`/`PoseStamped` 拆包、纯 `PositionCommand` 到 `PositionTarget` 的装包，以及话题、frame、mask 和 50 Hz 接口。

> **运行前提**：使用 **ROS Noetic (Py3.8)** 内核，并已 `source /opt/ros/noetic/setup.bash`、`source catkin_ws/devel/setup.bash`及 `catkin_make`。本 notebook 不初始化或运行节点，不发布任何话题，不调用 `spin()`。

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while root != root.parent and not (root / 'catkin_ws').is_dir():
    root = root.parent
scripts = root / 'catkin_ws' / 'src' / 'uav_tracking' / 'scripts'
assert scripts.is_dir(), '未找到 ROS 工作区'
sys.path.insert(0, str(scripts))

from geometry_msgs.msg import PoseStamped
from mavros_msgs.msg import PositionTarget
from uav_tracking.msg import Track as TrackMsg, TrackArray
import navigator_node as node
from navigator import on_timer as pure_on_timer

print('inputs :', node.TOPIC_TRACKS, '|', node.TOPIC_POSE)
print('output :', node.TOPIC_OUTPUT)
print('rate   : 50 Hz')
assert node.TOPIC_OUTPUT == '/ctrl/radar'

## 1. ROS 输入拆包为纯数据

`Track.msg -> Track`，`PoseStamped -> Pose`；节点壳不包含目标选择算法。

In [ ]:
array = TrackArray()
item = TrackMsg()
item.id, item.position, item.velocity = 4, [8.0, -2.0, 3.0], [0.2, 0.0, 0.0]
item.covariance, item.age = [0.0] * 36, 12
array.tracks.append(item)
pose_msg = PoseStamped()
pose_msg.pose.position.x, pose_msg.pose.position.y, pose_msg.pose.position.z = (1.0, 2.0, 5.0)
pose_msg.pose.orientation.w = 1.0
tracks = node.tracks_from_msg(array)
pose = node.pose_from_msg(pose_msg)
print('tracks:', tracks)
print('pose  :', pose)

## 2. `PositionCommand -> PositionTarget`

新鲜航迹由纯算法生成位置指令，节点只复制字段。输出是中间话题 `/ctrl/radar`，不是飞控出口。

In [ ]:
command = pure_on_timer(tracks, valid_age=0.0, last_cmd=None, current_pose=pose)
target = node.position_command_to_position_target(command)
print('pure command  :', command)
print('PositionTarget: frame=%d mask=%d position=(%.1f, %.1f, %.1f)' %
      (target.coordinate_frame, target.type_mask,
       target.position.x, target.position.y, target.position.z))
assert target.coordinate_frame == PositionTarget.FRAME_LOCAL_NED == 1
assert target.type_mask == node.POSITION_ONLY_TYPE_MASK == 3576
assert node.TOPIC_OUTPUT != '/mavros/setpoint_raw/local'

## 3. 空数组与超时悬停的转换

空 `TrackArray` 拆包为空列表；超时后纯算法用最新 `Pose` 生成悬停指令。

In [ ]:
empty_tracks = node.tracks_from_msg(TrackArray())
hover_command = pure_on_timer(empty_tracks, valid_age=0.51, last_cmd=command, current_pose=pose)
hover_target = node.position_command_to_position_target(hover_command)
print('empty tracks:', empty_tracks)
print('hover:', (hover_target.position.x, hover_target.position.y, hover_target.position.z))
assert (hover_target.position.x, hover_target.position.y, hover_target.position.z) == pose.position